In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.utils

>DOM utils 
output-file: core.dom.utils.html
title: core.dom.utils

This notebook provides utility functions for DOM manipulation.
---

In [ ]:
# | default_exp core.dom.utils

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

InteractiveShell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.basics import patch
from fastcore.test import *


In [ ]:
# | export
import asyncio
import os
import re
from pathlib import Path

import markdown


In [ ]:
#| export
from ollama import AsyncClient, ChatResponse, chat
from openai import AsyncOpenAI


In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | export
def get_text_summary_response(content: str, model:str="gemma3-27b", role:str="user", lang: str="zh") -> ChatResponse:
    """Generate a text-summary chat response for the given content.
    
    Args:
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            content = re.sub(r"\s+", " ", content.strip())
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    return chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
            }
        ],
    )

In [ ]:
# | export
image_link_pattern = r'[^\s]+\.(?:jpg|jpeg|png|gif|bmp|webp)'

def get_image_summary_response(image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> ChatResponse:
    """Generate an image-summary chat response for a local image path.
    
    Args:
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    if not re.match(image_link_pattern, image_link):
        # If the image link is not a URL, throw an error
        raise ValueError(f"Invalid image link: {image_link}")
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。关于机器人机械尺寸,运动范围,自由度的说明应简明扼要。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")

    response = chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
                'images': [f"{image_link}"],
            }
        ],
    )
    return response


In [ ]:
# | export
async def get_text_summary_response_async(client: AsyncClient, content: str, model:str="gemma3-27b", role:str="user", lang: str="zh") -> ChatResponse:
    """Generate a text-summary chat response with an async Ollama client.
    
    Args:
        client: Async Ollama client used to send the request.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    content = re.sub(r"\s+", " ", content.strip())
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    message = {
        "role": role,
        "content": prompt,
    }
    response =  await client.chat(
        model=model,
        messages=[message],
    )
    return response

In [ ]:
# | export
image_link_pattern = r'[^\s]+\.(?:jpg|jpeg|png|gif|bmp|webp)'

async def get_image_summary_response_async(client: AsyncClient, image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> ChatResponse:
    """Generate an image-summary chat response with an async Ollama client.
    
    Args:
        client: Async Ollama client used to send the request.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        ChatResponse: Raw model response containing the summary.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    # if not re.match(image_link_pattern, image_link):
    #     # If the image link is not a URL, throw an error
    #     raise ValueError(f"Invalid image link: {image_link}")
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。关于机器人机械尺寸,运动范围,自由度的说明应简明扼要。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")

    response = await client.chat(
        model=model,
        messages=[
            {
                "role": role,
                "content": prompt,
                'images': [f"{image_link}"],
            }
        ],
    )
    return response


In [ ]:
# | export
async def get_text_summary_response_openai_async(client: AsyncOpenAI, content: str, model:str="qwen-plus-2025-07-28", role:str="user", lang: str="zh") -> str:
    """Generate a text summary with an async OpenAI-compatible client.
    
    Args:
        client: Async OpenAI-compatible client used to send the request.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    match lang:
        case "en":
            prompt = (
                f"Please provide a summary of the following string. "
                f"The summary should be concise and informative: {content}. "
            )
        case "zh":
            content = re.sub(r"\s+", " ", content.strip())
            prompt = (
                f"请提供以下字符串的摘要。"
                f"摘要应简明扼要且信息丰富: {content}. "
            )
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": role, "content": prompt}],
        extra_headers={"X-RateLimit-RPM": "60", "X-RateLimit-TPM": "1000000"}
    )
    
    return response.choices[0].message.content or ""

In [ ]:
# | export
async def get_image_summary_response_openai_async(client: AsyncOpenAI, image_link: str | Path, model:str="qwen-vl-plus-2025-01-26", role:str="user", lang: str='zh') -> str:
    """Generate an image summary with an async OpenAI-compatible client.
    
    Args:
        client: Async OpenAI-compatible client used to send the request.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(image_link, Path):
        image_link = str(image_link)
    if not re.search(image_link_pattern, image_link):
        raise ValueError(f"Invalid image link: {image_link}")
    
    # Read and encode image as base64
    import base64
    with open(image_link, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    match lang:
        case 'en':
            prompt = "Please provide a summary of the following image. The summary should be concise and informative about the robot."
        case 'zh':
            prompt = "请提供以下图像的摘要。摘要应简明扼要，信息丰富，描述所显示的机器人内容。"
        case _:
            raise ValueError(f"Unsupported language: {lang}")
    
    response = await client.chat.completions.create(
        model=model,
        messages=[{
            "role": role,
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }], # type: ignore
        extra_headers={"X-RateLimit-RPM": "60", "X-RateLimit-TPM": "1000000"}
    )
    
    return response.choices[0].message.content or ""

In [ ]:
# | export
async def get_summary_response_async(client: AsyncClient | AsyncOpenAI, content: str, model:str="gemma3:27b", role:str="user", lang: str="zh") -> str:
    """Route text summarization to the matching async client implementation.
    
    Args:
        client: Async Ollama or OpenAI-compatible client.
        content: Source text to summarize.
        model: Chat model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(client, AsyncClient):
        # Ollama client
        response = await get_text_summary_response_async(client, content, model, role, lang)
        return response.message.content if hasattr(response, 'message') and hasattr(response.message, 'content') else "" # type: ignore
    elif isinstance(client, AsyncOpenAI):
        # OpenAI client
        return await get_text_summary_response_openai_async(client, content, model, role, lang)
    else:
        raise ValueError(f"Unsupported client type: {type(client)}")  # noqa: TRY004


In [ ]:
# | export
async def get_image_summary_async(client: AsyncClient | AsyncOpenAI, image_link: str | Path, model:str="gemma3:27b", role:str="user", lang: str='zh') -> str:
    """Route image summarization to the matching async client implementation.
    
    Args:
        client: Async Ollama or OpenAI-compatible client.
        image_link: Image file path or path-like string.
        model: Vision model name to use.
        role: Message role sent to the model.
        lang: Prompt language code.
    
    Returns:
        str: Extracted summary text.
    """
    if isinstance(client, AsyncClient):
        # Ollama client
        response = await get_image_summary_response_async(client, image_link, model, role, lang)
        return response.message.content if hasattr(response, 'message') and hasattr(response.message, 'content') else "" # type: ignore
    elif isinstance(client, AsyncOpenAI):
        # OpenAI client  
        return await get_image_summary_response_openai_async(client, image_link, model, role, lang)
    else:
        raise ValueError(f"Unsupported client type: {type(client)}")  # noqa: TRY004


In [ ]:
# | export

# OpenAI client for text summarization 
openai_client = AsyncOpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# Ollama client for embeddings and image analysis
ollama_client = AsyncClient()

# For backward compatibility, keep chat_client pointing to OpenAI client
chat_client = openai_client

In [ ]:
#| hide

# os.getcwd()
image_link = "../res/siasun_md_sammple_hrsl/SN024002/img/__2.png"
# image_link
res = re.match(image_link_pattern, image_link)
res.group(0)
# get_image_summary_response("../res/")

In [ ]:
imagepath = Path(os.getcwd()).parent / 'res/siasun_md_sample/SN024002/img/img_1.png'
# imagepath = Path(os.getcwd()).parent / 'res/siasun_md_sample/SN024002/img/img_13.png'
# imagepath = Path(os.getcwd()).parent / 'siasun_md_sample_hrsl/SN024002/img/img_11.png'
# imagepath = Path(os.getcwd()).parent / 'res/siasun_md_sample_hrsl/SX322002/img/img_13.png'
image_link = str(imagepath)
image_link
display(Markdown(f"![image]({image_link})"))

In [ ]:
# res = re.match(image_link_pattern, 'http://baidu.com/?home/img/small.png')
res = re.match(image_link_pattern, 'img/small.png')
# res = re.match(image_link_pattern, image_link)
# in case of match, print the matched string
if res:
    print(f"Matched image link: {res.group(0)}")
else:
    print(res)

In [ ]:
if True: 
    # response = await get_image_summary_response_async(image_link, model="gemma3:27b", role="user", lang='zh')

    # chat_client = AsyncClient(host="172.27.74.16:11434")
    chat_client = AsyncClient()
    response = await get_image_summary_response_async(chat_client, image_link, model="gemma3:4b", role="user", lang='en')
    assert isinstance(response.message.content, str), "Response content should be a string"
    md_text = markdown.markdown(response.message.content)
    Markdown(md_text)
    # print(md_text)
    # print(response.content)
    # response_txt = await get_text_summary_response_async(md_text,model="gemma3:27b", role="user", lang='zh')
    # md_text = markdown.markdown(response_txt.message.content)
    # Markdown(md_text)gte

In [ ]:
import importlib.util
import os
import shutil
import sys

print(f"sys.executable: {sys.executable}")
print(f"VIRTUAL_ENV: {os.environ.get('VIRTUAL_ENV')}")
print(f"which python: {shutil.which('python')}")
ollama_spec = importlib.util.find_spec('ollama')
print(f"ollama importable: {ollama_spec is not None}")
if ollama_spec is not None:
    import ollama
    print(f"ollama module: {getattr(ollama, '__file__', 'unknown')}")

In [ ]:
#| export


model = "deepseek-v4-flash:cloud"  # "deepseek-r1:32b" "gemma3:27b"
if not ("cloud" in model):
    chat_client: AsyncClient = AsyncClient()   
else:
    chat_client = AsyncClient(
        host="http://ollama.com",
        headers={"Authorization": f"Bearer {os.environ.get('OLLAMA_API_KEY')}"}
        )
msg_queue: asyncio.Queue = asyncio.Queue(maxsize=8)  # Limit the queue size to 8 messages
resp_queue: asyncio.Queue = asyncio.Queue(maxsize=1)  # Limit the queue size to 1 message for response 
lock = asyncio.Lock()  # Lock to ensure thread-safe access to the queues

In [ ]:
#| export


model = "qwen-plus-2025-07-28"  #  "deepseek-v3.2"  #  
chat_client = AsyncOpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

## Utils Tests

Fastcore-style checks for the exported utility functions in the generated module.

In [ ]:
# | hide
import json
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import ribosome.core.dom.utils as utils_mod


def make_chat_response(content: str):
    return SimpleNamespace(message=SimpleNamespace(content=content))


async def fake_async_chat(**kwargs):
    messages = kwargs["messages"]
    prompt = messages[0]["content"]
    if isinstance(prompt, list):
        prompt = prompt[0]["text"]
    return make_chat_response(f"RESP<{prompt}>")


async def fake_openai_create(**kwargs):
    messages = kwargs["messages"]
    content = messages[0]["content"]
    if isinstance(content, list):
        text = content[0]["text"]
    else:
        text = content
    return SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content=f"OPENAI<{text}>"))]
    )

In [ ]:
# | hide
with patch.object(utils_mod, "chat", return_value=make_chat_response("SYNC<TEXT>")):
    response = utils_mod.get_text_summary_response("  hello\nworld  ", lang="zh")
test_eq(response.message.content, "SYNC<TEXT>")

with patch.object(utils_mod, "chat", return_value=make_chat_response("SYNC<IMAGE>")):
    image_response = utils_mod.get_image_summary_response("img/test.png", lang="en")
test_eq(image_response.message.content, "SYNC<IMAGE>")

image_error = None
try:
    utils_mod.get_image_summary_response("not-an-image.txt")
except ValueError as exc:
    image_error = exc
test_eq("Invalid image link" in str(image_error), True)

lang_error = None
try:
    utils_mod.get_text_summary_response("hello", lang="fr")
except ValueError as exc:
    lang_error = exc
test_eq("Unsupported language" in str(lang_error), True)

ollama_client = utils_mod.AsyncClient()
ollama_client.chat = fake_async_chat

text_async = await utils_mod.get_text_summary_response_async(ollama_client, "  hello\nworld  ")
test_eq(text_async.message.content.startswith("RESP<请提供以下字符串的摘要。摘要应简明扼要且信息丰富: hello world. >"), True)

image_async = await utils_mod.get_image_summary_response_async(ollama_client, Path("img/test.png"), lang="en")
test_eq(image_async.message.content.startswith("RESP<Please provide a summary of the following image."), True)

openai_client = utils_mod.AsyncOpenAI(api_key="test", base_url="http://example.com")
openai_client.chat = SimpleNamespace(completions=SimpleNamespace(create=fake_openai_create))

openai_text = await utils_mod.get_text_summary_response_openai_async(openai_client, "  hello\nworld  ")
test_eq(openai_text.startswith("OPENAI<请提供以下字符串的摘要。摘要应简明扼要且信息丰富: hello world. >"), True)

tmpdir = tempfile.TemporaryDirectory()
image_path = Path(tmpdir.name) / "robot.jpg"
image_path.write_bytes(b"fake-image")
openai_image = await utils_mod.get_image_summary_response_openai_async(openai_client, image_path)
test_eq(openai_image.startswith("OPENAI<请提供以下图像的摘要。摘要应简明扼要，信息丰富，描述所显示的机器人内容。>"), True)

router_text_ollama = await utils_mod.get_summary_response_async(ollama_client, "hello")
test_eq(router_text_ollama.startswith("RESP<请提供以下字符串的摘要。"), True)

router_text_openai = await utils_mod.get_summary_response_async(openai_client, "hello")
test_eq(router_text_openai.startswith("OPENAI<请提供以下字符串的摘要。"), True)

router_image_ollama = await utils_mod.get_image_summary_async(ollama_client, image_path)
test_eq(router_image_ollama.startswith("RESP<请提供以下图像的摘要。"), True)

router_image_openai = await utils_mod.get_image_summary_async(openai_client, image_path)
test_eq(router_image_openai.startswith("OPENAI<请提供以下图像的摘要。"), True)

bad_client_error = None
try:
    await utils_mod.get_summary_response_async(object(), "hello")
except ValueError as exc:
    bad_client_error = exc
test_eq("Unsupported client type" in str(bad_client_error), True)

bad_image_client_error = None
try:
    await utils_mod.get_image_summary_async(object(), image_path)
except ValueError as exc:
    bad_image_client_error = exc
test_eq("Unsupported client type" in str(bad_image_client_error), True)

tmpdir.cleanup()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()